In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell if using Google Colab
# If you're running locally in VS Code, SKIP this cell entirely
# ============================================================

# !pip install langchain langchain-groq langchain-community langchain-chroma \
#              sentence-transformers chromadb faiss-cpu pypdf rank-bm25 \
#              python-dotenv requests -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# print("Colab setup complete!")

# Day 4 — M1: Embeddings + RAG Deep Dive 🧠

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Generate real embeddings and prove that similar meanings produce similar numbers
2. Compare all 4 chunking strategies on real text
3. Build a working vector database with ChromaDB
4. Build a complete end-to-end RAG pipeline from a PDF to an LLM answer
5. See RAG failure points live — and fix them
6. Add reranking to fix the 'wrong chunk at top' problem
7. Launch your multilingual RAG mini-project

**Time:** 3 hours (Marathon session) | **Difficulty:** Intermediate | **Prerequisites:** Day 1 + Day 2 complete

---

## 0. Setup — Load API Keys and Verify Packages

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key   = os.getenv("GROQ_API_KEY", "")
gemini_key = os.getenv("GOOGLE_API_KEY", "")

print(f"Groq API Key:    {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing — check .env file'}")
print(f"Gemini API Key:  {'✅ Loaded' if len(gemini_key) > 10 else '⚠️  Optional — not critical today'}")

# Verify today's key packages
packages = ["sentence_transformers", "chromadb", "langchain", "pypdf", "rank_bm25"]
for pkg in packages:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — run: pip install {pkg.replace('_', '-')}")

---
## 1. 🔢 Embeddings — Text as Numbers

From the theory: an embedding converts text into a list of numbers such that **similar meanings produce similar numbers**.

Let's prove that right now with real code.

### 1.1 Your first embedding — see the 384 numbers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model — first time downloads ~90MB and caches locally
# After that it loads from cache instantly — no internet needed
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

# Embed a single sentence
sentence = "New employees get 12 days of casual leave per year."
embedding = model.encode(sentence)

print(f"\nSentence: '{sentence}'")
print(f"Embedding shape: {embedding.shape}")      # should be (384,)
print(f"First 10 numbers: {embedding[:10].round(4)}")
print(f"Last  10 numbers: {embedding[-10:].round(4)}")
print(f"\nMin value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")
print(f"Mean:      {embedding.mean():.4f}")

### 1.2 Prove it — similar meanings produce similar numbers

This is the core claim. Let's test it with real sentences.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sentences = [
    "I love dogs.",                             # 0
    "I adore puppies.",                         # 1  ← should be close to 0
    "Dogs are my favourite animals.",           # 2  ← should be close to 0 and 1
    "The stock market crashed yesterday.",      # 3  ← should be FAR from 0,1,2
    "Equity prices fell sharply this morning.", # 4  ← should be close to 3
    "Agentic AI is transforming software.",     # 5  ← unrelated to both clusters
]

embeddings = model.encode(sentences)
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix (1.0 = identical meaning, 0.0 = unrelated)")
print("=" * 65)
labels = ["dogs", "puppies", "fav animals", "stock crash", "equity fell", "agentic AI"]
header = f"{'':15}" + "".join(f"{l:>13}" for l in labels)
print(header)
print("-" * 95)
for i, row in enumerate(sim_matrix):
    print(f"{labels[i]:15}" + "".join(f"{v:>13.3f}" for v in row))

print("\n--- Key Observations ---")
print(f"'dogs' ↔ 'puppies':       {sim_matrix[0][1]:.3f}  ← very similar (should be ~0.8+)")
print(f"'dogs' ↔ 'stock crash':   {sim_matrix[0][3]:.3f}  ← very different (should be ~0.0-0.2)")
print(f"'stock crash' ↔ 'equity': {sim_matrix[3][4]:.3f}  ← similar topic (should be ~0.7+)")

### 1.3 Tokens vs Embeddings — side by side

Quick reminder from Day 1 — tokens are lookup numbers. Embeddings capture meaning.

In [ ]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

words = ["King", "Queen", "Car"]

print("TOKEN IDs — just lookup numbers, no meaning")
print("-" * 45)
for word in words:
    token_ids = encoder.encode(word)
    print(f"  '{word}' → token IDs: {token_ids}")

print("\nEMBEDDINGS — 384 numbers that capture meaning")
print("-" * 45)
word_embeddings = model.encode(words)
for i, word in enumerate(words):
    print(f"  '{word}' → [{word_embeddings[i][0]:.3f}, {word_embeddings[i][1]:.3f}, {word_embeddings[i][2]:.3f}, ...]")

# Similarity between King, Queen, Car
sims = cosine_similarity(word_embeddings)
print(f"\n  King ↔ Queen similarity: {sims[0][1]:.3f}  ← both royalty, similar meaning")
print(f"  King ↔ Car   similarity: {sims[0][2]:.3f}  ← completely unrelated")

print("\nKEY INSIGHT: Token IDs are roll numbers. Embeddings are personality profiles.")

---
## 2. ✂️ Chunking — 4 Strategies Compared

Before we can embed a document, we need to split it into manageable pieces.
Let's try all 4 strategies on the same text and see the difference.

In [ ]:
# Sample document — HR Policy excerpt
SAMPLE_DOC = """
Employee Leave Policy

New employees are entitled to 12 days of casual leave per calendar year. 
This entitlement begins from the date of joining. Employees must apply for 
casual leave at least 3 days in advance using the HR portal.

Medical leave is separate from casual leave. Employees can avail up to 15 
days of medical leave per year. A doctor's certificate is mandatory for any 
medical leave exceeding 2 consecutive days.

Leave carry forward rules vary by employee grade. Junior employees (Grade 1-3) 
can carry forward up to 5 days of unused casual leave to the next year. Senior 
employees (Grade 4 and above) can carry forward up to 10 days.

Unused leaves that are not carried forward can be encashed at the end of the 
financial year. The encashment rate is calculated based on the employee's 
current basic salary divided by 30 days.

Office timings are 9:00 AM to 6:00 PM, Monday through Friday. Saturday and 
Sunday are official holidays. Employees working on weekends are entitled to 
compensatory off as per the overtime policy.
"""

print(f"Sample document: {len(SAMPLE_DOC)} characters, {len(SAMPLE_DOC.split())} words")

### 2.1 Strategy 1 — Fixed Size (CharacterTextSplitter)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    chunk_size=200,        # max characters per chunk
    chunk_overlap=0,       # no overlap
    separator="",          # split anywhere — no respect for words or sentences
)

fixed_chunks = fixed_splitter.split_text(SAMPLE_DOC)

print(f"Strategy 1 — Fixed Size: {len(fixed_chunks)} chunks")
print("=" * 60)
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(repr(chunk))
    
print("\n⚠️  Notice: chunks can cut mid-word or mid-sentence!")

### 2.2 Strategy 2 — Sentence-Based (NLTK)

In [ ]:
# Simple sentence splitter using Python — NLTK version requires download
# We'll use a clean regex-based approach for the demo
import re

def sentence_splitter(text, sentences_per_chunk=2):
    """Split text into chunks of N sentences each."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i+sentences_per_chunk])
        chunks.append(chunk)
    return chunks

sentence_chunks = sentence_splitter(SAMPLE_DOC, sentences_per_chunk=2)

print(f"Strategy 2 — Sentence-Based: {len(sentence_chunks)} chunks")
print("=" * 60)
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(chunk)

print("\n✅ No mid-word cuts. But chunks are very small — limited context.")

### 2.3 Strategy 3 — Recursive (DEFAULT — use this for most projects)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# This is the recommended default for 90% of RAG projects
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,         # target size in characters
    chunk_overlap=60,       # overlap between consecutive chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # tries these in order
)

recursive_chunks = recursive_splitter.split_text(SAMPLE_DOC)

print(f"Strategy 3 — Recursive: {len(recursive_chunks)} chunks")
print("=" * 60)
for i, chunk in enumerate(recursive_chunks[:4]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(chunk)

print("\n✅ Respects paragraphs first, then sentences. Best balance.")

# Show the overlap between consecutive chunks
print("\n--- Overlap demo (end of Chunk 1 / start of Chunk 2) ---")
if len(recursive_chunks) >= 2:
    print(f"End of Chunk 1:   '...{recursive_chunks[0][-80:]}'")
    print(f"Start of Chunk 2: '{recursive_chunks[1][:80]}...'")

### 2.4 Strategy 4 — Semantic (highest quality, slowest)

In [ ]:
# Semantic chunking — splits when the topic CHANGES
# We'll implement the core idea manually to understand it
# (langchain_experimental.SemanticChunker requires extra install)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunker(text, threshold=0.4):
    """Split text at topic boundaries using embedding similarity."""
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 15]
    
    if len(sentences) < 2:
        return sentences
    
    # Embed all sentences
    embeddings = model.encode(sentences)
    
    # Find similarity between consecutive sentences
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        
        if sim < threshold:  # topic changed — start new chunk
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:                # same topic — continue current chunk
            current_chunk.append(sentences[i])
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

semantic_chunks = semantic_chunker(SAMPLE_DOC, threshold=0.4)

print(f"Strategy 4 — Semantic: {len(semantic_chunks)} chunks")
print("=" * 60)
for i, chunk in enumerate(semantic_chunks):
    print(f"\n[Chunk {i+1}] (topic group):")
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

print("\n✅ Each chunk is a semantically coherent topic. Quality over speed.")

### 2.5 Strategy Comparison — side by side

In [ ]:
print("Chunking Strategy Comparison")
print("=" * 65)
print(f"{'Strategy':<22} {'Chunks':>7} {'Avg Size':>10} {'Respects Words?':>17}")
print("-" * 65)

strategies = [
    ("1. Fixed Size",     fixed_chunks,     "❌ No"),
    ("2. Sentence-Based", sentence_chunks,  "✅ Yes"),
    ("3. Recursive",      recursive_chunks, "✅ Yes (DEFAULT)"),
    ("4. Semantic",       semantic_chunks,  "✅ Yes (best)"),
]

for name, chunks, respects in strategies:
    avg_size = int(np.mean([len(c) for c in chunks]))
    print(f"{name:<22} {len(chunks):>7} {avg_size:>10} chars {respects:>17}")

print("\n💡 For this course: always use Strategy 3 — RecursiveCharacterTextSplitter")

---
## 3. 🗄️ Vector Database — ChromaDB

Now we have chunks. Let's embed them and store them in a vector database so we can search by meaning.

### 3.1 Build a mini ChromaDB collection

In [ ]:
import chromadb

# Create an in-memory ChromaDB client
# For persistence, use: chromadb.PersistentClient(path="./chroma_db")
client = chromadb.Client()

# Create a collection — like a table in a regular database
collection = client.create_collection(
    name="hr_policy",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

# Use our recursive chunks — they're the best quality
chunks = recursive_chunks

# Add chunks to ChromaDB
# ChromaDB needs: documents (text), ids (unique), and optional metadata
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[
        {"source": "HR_Policy.pdf", "chunk_index": i, "char_count": len(chunk)}
        for i, chunk in enumerate(chunks)
    ]
)

print(f"✅ Added {collection.count()} chunks to ChromaDB")
print(f"\nCollection name: {collection.name}")
print(f"Total documents: {collection.count()}")

### 3.2 Query the vector database by meaning

In [ ]:
# Query 1 — leave days for new employees
query1 = "How many leave days do new employees get?"

results1 = collection.query(
    query_texts=[query1],
    n_results=3  # return top 3 most similar chunks
)

print(f"Query: '{query1}'")
print("=" * 65)
for i, (doc, meta, dist) in enumerate(zip(
    results1['documents'][0],
    results1['metadatas'][0],
    results1['distances'][0]
)):
    similarity = 1 - dist  # ChromaDB returns distance, convert to similarity
    print(f"\n[Rank {i+1}] Similarity: {similarity:.3f}")
    print(f"Source: {meta['source']}, Chunk index: {meta['chunk_index']}")
    print(f"Text: {doc[:200]}")

In [ ]:
# Query 2 — carry forward policy
query2 = "Can I carry forward my unused leaves to next year?"

results2 = collection.query(query_texts=[query2], n_results=3)

print(f"Query: '{query2}'")
print("=" * 65)
for i, (doc, dist) in enumerate(zip(
    results2['documents'][0],
    results2['distances'][0]
)):
    print(f"\n[Rank {i+1}] Similarity: {1 - dist:.3f}")
    print(f"Text: {doc[:200]}")

### 3.3 Metadata filtering — the ChromaDB superpower

In [ ]:
# Add chunks from a SECOND document to the same collection
second_doc_chunks = [
    "The Performance Review Policy states that all employees undergo annual reviews.",
    "Employees receiving a rating of 'Exceeds Expectations' are eligible for a 15% bonus.",
    "Performance Improvement Plans (PIPs) are issued to employees rated 'Below Expectations'.",
]

collection.add(
    documents=second_doc_chunks,
    ids=[f"perf_chunk_{i}" for i in range(len(second_doc_chunks))],
    metadatas=[{"source": "Performance_Policy.pdf", "chunk_index": i} for i in range(len(second_doc_chunks))]
)

print(f"✅ Total chunks in collection: {collection.count()}")

# Now search WITHOUT filter — gets results from both documents
query = "What are the rules for employees?"
all_results = collection.query(query_texts=[query], n_results=3)
print(f"\nSearch '{query}' (no filter) — results from both docs:")
for doc, meta in zip(all_results['documents'][0], all_results['metadatas'][0]):
    print(f"  [{meta['source']}] {doc[:80]}...")

# Search WITH metadata filter — only HR_Policy.pdf
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"source": "HR_Policy.pdf"}  # ← metadata filter
)
print(f"\nSearch '{query}' (filtered to HR_Policy.pdf only):")
for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(f"  [{meta['source']}] {doc[:80]}...")

print("\n💡 Metadata filtering is powerful for multi-document RAG systems!")

---
## 4. 🏗️ Full RAG Pipeline — End to End

Now let's put everything together into a complete RAG pipeline that:
1. Takes a document
2. Chunks + embeds + stores it
3. Takes a user question
4. Retrieves relevant chunks
5. Sends to LLM with a proper prompt
6. Returns a grounded answer

### 4.1 The RAG prompt template — production-ready

In [ ]:
def build_rag_prompt(question: str, chunks: list[str]) -> str:
    """Build a production-ready RAG prompt.
    
    3 Golden Rules:
    1. Answer ONLY from context
    2. Say I don't know if answer not in context
    3. Label chunks clearly [Chunk N]
    """
    context = ""
    for i, chunk in enumerate(chunks):
        context += f"[Chunk {i+1}]: {chunk}\n\n"
    
    prompt = f"""You are a helpful assistant. Answer the user's question ONLY using \
the context provided below.
If the answer is not found in the context, respond with exactly:
"I don't have enough information to answer this question."
Do NOT use your own knowledge. Do NOT make up information.

CONTEXT:
{context}
QUESTION:
{question}

ANSWER:"""
    
    return prompt

# Test it
test_chunks = [
    "New employees get 12 days of casual leave per year.",
    "Medical leave requires a doctor's certificate for more than 2 days.",
]
test_prompt = build_rag_prompt("How many days of leave do new employees get?", test_chunks)
print(test_prompt)

### 4.2 The complete RAG function

In [ ]:
from langchain_groq import ChatGroq

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

def rag_answer(
    question: str,
    collection,
    k: int = 3,
    source_filter: str = None,
    verbose: bool = True
) -> str:
    """Complete RAG pipeline: question → retrieve → prompt → LLM → answer."""
    
    # Step 1: Retrieve top-k chunks from vector DB
    query_params = {"query_texts": [question], "n_results": k}
    if source_filter:
        query_params["where"] = {"source": source_filter}
    
    results = collection.query(**query_params)
    chunks  = results['documents'][0]
    
    if verbose:
        print(f"📥 Question: {question}")
        print(f"🔍 Retrieved {len(chunks)} chunks:")
        for i, (chunk, dist) in enumerate(zip(chunks, results['distances'][0])):
            print(f"   [{i+1}] similarity={1-dist:.3f} — {chunk[:80]}...")
        print()
    
    # Step 2: Build RAG prompt
    prompt = build_rag_prompt(question, chunks)
    
    # Step 3: Send to LLM
    response = llm.invoke(prompt)
    
    return response.content

print("✅ RAG function ready!")

In [ ]:
# Test with real questions
questions = [
    "How many casual leave days do new employees get?",
    "Can I carry forward unused leaves to next year?",
    "What is the office timing?",
    "What is the salary of the CEO?",  # ← not in document — should say I don't know
]

print("=" * 65)
for q in questions:
    print()
    answer = rag_answer(q, collection, k=3, source_filter="HR_Policy.pdf")
    print(f"💬 Answer: {answer}")
    print("-" * 65)

---
## 5. 💥 RAG Failure Points — Live Demos

Understanding failure modes is as important as understanding how RAG works.
Let's trigger each failure deliberately and then fix it.

### 5.1 Failure 5 — LLM Hallucination (the most dangerous)

In [ ]:
# BAD PROMPT — no instruction to stay in context
def bad_rag_prompt(question, chunks):
    """The naive way — no ONLY from context instruction."""
    context = "\n".join(chunks)
    return f"{context}\n\nQuestion: {question}\nAnswer:"

# Ask about something NOT in our document
question = "What is the annual bonus percentage for employees?"
results = collection.query(query_texts=[question], n_results=3, where={"source": "HR_Policy.pdf"})
chunks = results['documents'][0]

print("=== WITHOUT proper prompt (hallucination risk) ===")
bad_prompt = bad_rag_prompt(question, chunks)
bad_answer = llm.invoke(bad_prompt)
print(f"Q: {question}")
print(f"A: {bad_answer.content}")

print("\n=== WITH proper prompt (grounded answer) ===")
good_answer = rag_answer(question, collection, k=3, source_filter="HR_Policy.pdf", verbose=False)
print(f"Q: {question}")
print(f"A: {good_answer}")

print("\n💡 The proper prompt makes the LLM admit it doesn't know instead of guessing!")

### 5.2 Failure 1 — Vocabulary Mismatch

In [ ]:
# The document uses 'casual leave'
# The user asks using 'time off' and 'vacation days'

user_phrasings = [
    "How many days of casual leave do new employees get?",  # exact match
    "How many vacation days do new employees get?",          # different word
    "How many days of time off do new employees have?",      # very different
]

print("Vocabulary Mismatch Test")
print("=" * 65)
print("Document uses: 'casual leave'")
print()

for q in user_phrasings:
    results = collection.query(query_texts=[q], n_results=1, where={"source": "HR_Policy.pdf"})
    top_chunk  = results['documents'][0][0]
    similarity = 1 - results['distances'][0][0]
    print(f"Query:      '{q}'")
    print(f"Similarity: {similarity:.3f}")
    print(f"Top chunk:  '{top_chunk[:80]}...'")
    print()

print("💡 Embeddings handle vocabulary mismatch BETTER than keyword search.")
print("   'vacation days' still finds 'casual leave' because meanings are similar.")

### 5.3 Failure 3 — Answer Spans Multiple Chunks (K too small)

In [ ]:
# This question needs information from TWO separate chunks
question = "What is the difference between casual leave and medical leave?"

print("K=1 (too small — misses one of the policies):")
print("-" * 55)
answer_k1 = rag_answer(question, collection, k=1, source_filter="HR_Policy.pdf")
print(f"Answer: {answer_k1}\n")

print("K=4 (better — captures both policies):")
print("-" * 55)
answer_k4 = rag_answer(question, collection, k=4, source_filter="HR_Policy.pdf")
print(f"Answer: {answer_k4}")

print("\n💡 For comparison questions, increase K to ensure both sides are retrieved.")

---
## 6. 🎯 Reranking — Fix the Wrong Chunk at Top Problem

Sometimes vector search retrieves the right general topic but the wrong specific chunk.
Reranking fixes this with a second, smarter filter.

### 6.1 The problem — similar topic, wrong answer

In [ ]:
# Add a chunk that is topically similar but answers a different question
# This simulates the 'encashment vs carry-forward' problem from class

confusing_chunks = [
    "Unused casual leaves can be encashed at the end of financial year at a rate of basic salary / 30 per day.",
    "Leave carry forward: Junior employees (Grade 1-3) can carry forward up to 5 days. Senior employees (Grade 4+) up to 10 days.",
    "Leave encashment is processed in March every year. Employees must apply by February 28th.",
    "The HR portal must be used to apply for leave carry forward by December 31st.",
]

# Create fresh collection for this demo
client2 = chromadb.Client()
demo_collection = client2.create_collection("rerank_demo", metadata={"hnsw:space": "cosine"})
demo_collection.add(
    documents=confusing_chunks,
    ids=[f"doc_{i}" for i in range(len(confusing_chunks))]
)

# Query about carry-forward
question = "Can I carry forward my unused leaves to next year?"

results = demo_collection.query(query_texts=[question], n_results=4)

print(f"Question: '{question}'")
print("\nVector Search Results (before reranking):")
print("=" * 65)
for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
    sim = 1 - dist
    marker = "← CORRECT" if "carry forward" in doc.lower() else "← misleading"
    print(f"\n[Rank {i+1}] Similarity: {sim:.3f}  {marker}")
    print(f"  {doc[:100]}")

print("\n⚠️  Notice: 'encashment' chunks may rank higher than 'carry forward' chunks!")

### 6.2 The fix — CrossEncoder reranking

In [ ]:
from sentence_transformers import CrossEncoder

# Load the reranker — free, local, ~80MB
print("Loading cross-encoder reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded!")

def rag_with_reranking(
    question: str,
    collection,
    k_retrieve: int = 10,  # retrieve more
    k_final: int = 3,       # keep only top after reranking
) -> str:
    """RAG pipeline with reranking — retrieve more, filter smarter."""
    
    # Step 1: Vector search — retrieve top k_retrieve
    results  = collection.query(query_texts=[question], n_results=k_retrieve)
    chunks   = results['documents'][0]
    
    # Step 2: Rerank — cross-encoder reads Q+chunk TOGETHER
    pairs    = [[question, chunk] for chunk in chunks]
    scores   = reranker.predict(pairs)  # direct relevance score — no comparison!
    
    # Step 3: Sort by reranker score and keep top k_final
    ranked   = sorted(zip(scores, chunks), key=lambda x: x[0], reverse=True)
    top_chunks = [chunk for _, chunk in ranked[:k_final]]
    
    print(f"Question: '{question}'")
    print(f"\nAfter reranking (top {k_final} from {k_retrieve} retrieved):")
    for i, (score, chunk) in enumerate(ranked[:k_final]):
        marker = "← CORRECT" if "carry forward" in chunk.lower() else ""
        print(f"  [Rank {i+1}] Reranker score: {score:.4f}  {marker}")
        print(f"    {chunk[:100]}")
    
    # Step 4: Build prompt and get LLM answer
    prompt   = build_rag_prompt(question, top_chunks)
    response = llm.invoke(prompt)
    return response.content

answer = rag_with_reranking("Can I carry forward my unused leaves to next year?", demo_collection)
print(f"\n💬 Final Answer: {answer}")

In [ ]:
# Side-by-side comparison — with vs without reranking
question = "Can I carry forward my unused leaves to next year?"

print("COMPARISON: With vs Without Reranking")
print("=" * 65)

# Without reranking — just top-3 from vector search
plain_results = demo_collection.query(query_texts=[question], n_results=3)
plain_chunks  = plain_results['documents'][0]
plain_prompt  = build_rag_prompt(question, plain_chunks)
plain_answer  = llm.invoke(plain_prompt).content

print("\n❌ Without Reranking:")
print(f"   {plain_answer}")

# With reranking
print("\n✅ With Reranking:")
pairs    = [[question, c] for c in plain_results['documents'][0] + 
            demo_collection.query(query_texts=[question], n_results=4)['documents'][0]]
scores   = reranker.predict(pairs)
all_chunks = plain_results['documents'][0] + demo_collection.query(query_texts=[question], n_results=4)['documents'][0]
ranked   = sorted(zip(scores, all_chunks), key=lambda x: x[0], reverse=True)
top3     = [c for _, c in ranked[:3]]
reranked_answer = llm.invoke(build_rag_prompt(question, top3)).content
print(f"   {reranked_answer}")

---
## 7. 🌍 LangChain RAG — The Clean Production Way

Everything above was built from scratch so you understand every piece.
Now let's see how LangChain wraps all of this into a clean 10-line pipeline.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Step 1: Embedding model (same all-MiniLM-L6-v2, wrapped for LangChain)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Step 2: Split our document
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60)
lc_chunks = splitter.create_documents(
    texts=[SAMPLE_DOC],
    metadatas=[{"source": "HR_Policy"}]
)
print(f"Created {len(lc_chunks)} chunks")

# Step 3: Create vector store (in-memory Chroma)
vectorstore = Chroma.from_documents(lc_chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"✅ Vector store ready")

# Step 4: RAG prompt
prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question ONLY using the context below.
If the answer is not in the context, say "I don't have enough information."

Context: {context}

Question: {question}

Answer:""")

# Step 5: LLM
lc_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Step 6: Chain everything together using LCEL (LangChain Expression Language)
def format_docs(docs):
    return "\n\n".join([f"[Chunk {i+1}]: {d.page_content}" for i, d in enumerate(docs)])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | lc_llm
    | StrOutputParser()
)

print("✅ LangChain RAG chain ready!")

In [ ]:
# Test the LangChain RAG chain
test_questions = [
    "How many casual leave days do new employees receive?",
    "What are the carry forward rules for senior employees?",
    "What time does the office open?",
]

for q in test_questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")
    print("-" * 60)

---
## 8. 🏆 Mini-Project — RAG in Your Mother Tongue

Everything you need is now in your hands. Your turn.

**The Challenge:** Build a complete RAG pipeline for a document in YOUR native language.

**What to do:**
1. Pick a document in your native language (Wikipedia article, government notice, news article)
2. Replace the one line for the embedding model
3. Run the complete pipeline
4. Ask questions in your native language
5. Note what works well and what breaks

**This is real unsolved territory** — you'll discover genuine problems that Indian AI companies are actively trying to fix.

In [ ]:
# ============================================================
# MINI-PROJECT SKELETON — Fill in your language + document
# ============================================================

# STEP 1: Choose your language
# Telugu | Hindi | Tamil | Kannada | Malayalam | Marathi | Bengali
MY_LANGUAGE = "Telugu"   # ← change this

# STEP 2: Add your document text
# Find a Wikipedia article or any text in your language
# Tip: Go to <your-language>.wikipedia.org and copy a short article
MY_DOCUMENT = """
PUT YOUR NATIVE LANGUAGE DOCUMENT HERE.

Example for Telugu — paste a Wikipedia article about Hyderabad, 
a news article, or any text of your choice.

Longer is better — at least 5-10 paragraphs for interesting retrieval.
"""

# STEP 3: Load the multilingual embedding model
# This is the ONLY change needed from English RAG!
print(f"Loading multilingual model for {MY_LANGUAGE}...")
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Multilingual model loaded — supports 50+ languages including all Indian languages!")

# STEP 4: Test the model on your language
test_sentence = "నమస్కారం"  # ← replace with a sentence in your language
test_embedding = multilingual_model.encode(test_sentence)
print(f"\nTest: '{test_sentence}' → embedding shape: {test_embedding.shape}")
print(f"First 5 values: {test_embedding[:5].round(4)}")
print("\n✅ Your language embeds correctly!")

In [ ]:
# STEP 5: Build the RAG pipeline for your language

from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual embedding model
multilingual_embeddings = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# Split your document
multi_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
multi_chunks = multi_splitter.create_documents(
    texts=[MY_DOCUMENT],
    metadatas=[{"source": f"{MY_LANGUAGE}_document", "language": MY_LANGUAGE}]
)

print(f"Document split into {len(multi_chunks)} chunks")

# Create vector store
multi_vectorstore = Chroma.from_documents(multi_chunks, multilingual_embeddings)
multi_retriever = multi_vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG chain (same as before — only embeddings changed)
multi_rag_chain = (
    {"context": multi_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | lc_llm
    | StrOutputParser()
)

print(f"✅ Multilingual RAG pipeline ready for {MY_LANGUAGE}!")
print("\n💡 Now ask questions in your native language ↓")

In [ ]:
# STEP 6: Ask questions in your native language!

# ← Replace with actual questions in your language
my_questions = [
    "Add your first question here in your native language",
    "Add your second question here",
    "Add a question whose answer is NOT in the document (to test I don't know)",
]

print(f"=== RAG in {MY_LANGUAGE} ===")
for q in my_questions:
    print(f"\nQ: {q}")
    answer = multi_rag_chain.invoke(q)
    print(f"A: {answer}")
    print("-" * 55)

### 8.1 🔬 Reflection — What did you discover?

After running your multilingual RAG, answer these questions in the markdown cell below.

**My Observations (fill this in):**

**Language I used:** ___

**Document I chose:** ___

**What worked well:**
- 
- 

**What broke or gave wrong answers:**
- 
- 

**Why do you think it broke? (based on what you learned today):**
- 

**One thing you would fix if you had more time:**
- 

---
## 9. Day 4 Wrap-up

### What you built today:
- ✅ Generated real embeddings and proved similar meanings → similar numbers
- ✅ Compared all 4 chunking strategies on real text
- ✅ Built a vector database with ChromaDB — with metadata filtering
- ✅ Built a complete end-to-end RAG pipeline from scratch
- ✅ Triggered RAG failures deliberately and fixed them
- ✅ Added reranking with CrossEncoder to fix the 'wrong chunk at top' problem
- ✅ Rebuilt the pipeline with LangChain LCEL — the clean production way
- ✅ Started your multilingual RAG mini-project

### The complete 10-step RAG pipeline:
```
1.  Load document
2.  Chunk  (RecursiveCharacterTextSplitter, size=400, overlap=80)
3.  Load embedding model  (all-MiniLM-L6-v2)
4.  Embed every chunk  → 384-dimensional vectors
5.  Store in ChromaDB  (vectors + text + metadata)
          ↓  (indexing done — above runs once)
6.  User asks a question
7.  Embed question  (SAME model)
8.  Cosine similarity search  → top K chunks
9.  Rerank with CrossEncoder  (optional but recommended)
10. Build prompt (ONLY from context rule) + LLM → grounded answer
```

### Tomorrow — Day 5: Advanced RAG
We go beyond basic RAG with **Hybrid Search** (BM25 + vector search combined),
**Parent-Child chunking**, **HyDE** (Hypothetical Document Embeddings),
and **RAG Evaluation** with RAGAS.

---

### 📚 Want to explore more?
- [Sentence Transformers docs](https://www.sbert.net)
- [ChromaDB docs](https://docs.trychroma.com)
- [LangChain RAG guide](https://python.langchain.com/docs/use_cases/question_answering)
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — compare embedding models
- [AI4Bharat](https://ai4bharat.iitm.ac.in) — Indian language AI research